In [ ]:
from statsbombpy import sb
import pandas as pd
import numpy as np

import warnings

pd.set_option("display.max_columns", None)

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# 1. Fetch the 2003/2004 Premier League season (competition_id=2, season_id=44)
matches = sb.matches(competition_id=2, season_id=44)

# 2. Filter for Arsenal matches
arsenal_matches = matches[
    (matches["home_team"] == "Arsenal") | (matches["away_team"] == "Arsenal")
]

# 3. Grab the match_id for Arsenal vs. Henry's masterclass opponents
match_id = arsenal_matches.iloc[0]["match_id"]
match_name = (
    f"{arsenal_matches.iloc[0]['home_team']} vs {arsenal_matches.iloc[0]['away_team']}"
)
print(f"Fetching tick data for: {match_name}\n")

# 4. Pull the high-frequency event tick data
events = sb.events(match_id=match_id)
# 5. Do some cleaning
events["x"] = events["location"].apply(lambda x: x[0] if isinstance(x, list) else None)
events["y"] = events["location"].apply(lambda x: x[1] if isinstance(x, list) else None)

# --- Fix timestamps because it currently has the time resetting from 00:00 at halftime
# 1. Convert the relative timestamp into total seconds for that specific period
events["period_seconds"] = pd.to_timedelta(events["timestamp"]).dt.total_seconds()

# 2. Sort to guarantee strict chronological order
events = events.sort_values(by=["period", "period_seconds"]).reset_index(drop=True)

# 3. Calculate holding time by grouping by period.
# This naturally prevents calculating a transition across the halftime whistle.
events["delta_t"] = events.groupby("period")["period_seconds"].diff()

# The first event of any period will naturally yield a NaN delta_t.
events["delta_t"] = events["delta_t"].fillna(0)

# 1. Drop administrative events by requiring valid [x, y] coordinates
# This removes 'Starting XI', 'Half Start', 'Half End', etc.
game_states = events.dropna(subset=["x", "y"]).copy()

# 2. Re-sort just to be absolutely certain of chronological integrity
game_states = game_states.sort_values(by=["period", "period_seconds"]).reset_index(
    drop=True
)

# 3. NOW calculate your holding times on the clean spatial data
game_states["delta_t"] = game_states.groupby("period")["period_seconds"].diff()
game_states["delta_t"] = game_states["delta_t"].fillna(0)

print(game_states[["type", "period_seconds", "delta_t", "x", "y"]].head())

In [ ]:
print(game_states.head())

In [ ]:
# --- Experimenting Notebook ---
events_35th_min = events[events["minute"] == 35].copy()
# print(events_35th_min.head())

shots_df = events[events["type"] == "Shot"].copy()
print(shots_df[shots_df["shot_outcome"] == "Goal"])

# print(sum(sum(events["location"] == np.nan)))
loc_df = events.loc[:, "location"]
# print(sum(loc_df.isna()))
# print(events[events["location"].isna() == True])  # count how many NaNs 'location' has
# see if any location coordinates are outside [0, 120]
# print(sum(events["x"] > 120))
# print(sum(events["y"] > 80))
# print(sum(events["x"] < 0))
# print(sum(events["y"] < 0))

In [ ]:
import numpy as np

# 1. Define the grid edges
# 6 zones across the 120 length = 20-unit increments
x_edges = np.linspace(0, 120, 7)
# 4 zones across the 80 width = 20-unit increments
y_edges = np.linspace(0, 80, 5)

# 2. Map coordinates to X and Y bins
# labels=False returns the integer index of the bin (0, 1, 2...)
game_states["x_bin"] = pd.cut(
    game_states["x"], bins=x_edges, labels=False, include_lowest=True
)
game_states["y_bin"] = pd.cut(
    game_states["y"], bins=y_edges, labels=False, include_lowest=True
)

# 3. Create a single, unique Zone ID (from 0 to 23)
# Formula: (x_bin * number_of_y_bins) + y_bin
game_states["zone_id"] = (game_states["x_bin"] * 4) + game_states[
    "y_bin"
]  # pitch split into a grid

print(game_states[["type", "x", "y", "x_bin", "y_bin", "zone_id"]].head(10))

In [ ]:
# Assuming your target team for the engine is 'Arsenal'
target_team = "Arsenal"

# Create a binary possession state (1 for target team, 0 for opponent)
# We cast the boolean to an integer for the Markov transition matrix
game_states["P"] = (game_states["possession_team"] == target_team).astype(int)

# Now, let's look at your finalized State Vector components:
print(
    game_states[
        ["type", "minute", "second", "team", "possession_team", "P", "zone_id"]
    ].head(10)
)

In [ ]:
# 1. Identify valid goals (A 'Shot' where the outcome is 'Goal')
# Note: For V1.0, we are ignoring the 'Own Goal' event type to keep the pipeline clean,
# but you will need to add it in V2.0.
is_goal = (game_states["type"] == "Shot") & (game_states["shot_outcome"] == "Goal")

# 2. Create binary event flags for who scored the goal at that exact tick
game_states["arsenal_goal_event"] = np.where(
    is_goal & (game_states["team"] == target_team), 1, 0
)
game_states["opponent_goal_event"] = np.where(
    is_goal & (game_states["team"] != target_team), 1, 0
)

# 3. Use cumulative sum to maintain the running score throughout the match
game_states["arsenal_score"] = game_states["arsenal_goal_event"].cumsum()
game_states["opponent_score"] = game_states["opponent_goal_event"].cumsum()

# 4. Calculate Delta G (Arsenal Goals - Opponent Goals)
game_states["delta_G"] = game_states["arsenal_score"] - game_states["opponent_score"]
# 5. Cap the Goal Difference to prevent matrix sparsity
game_states["delta_G"] = game_states["delta_G"].clip(-2, 2)

In [ ]:
# Create a unique state identifier: e.g., "G:1_P:1_Z:14"
# (Up by 1 goal, Arsenal has Possession, Ball is in Zone 14)
game_states["state_id"] = (
    "G:"
    + game_states["delta_G"].astype(str)
    + "_P:"
    + game_states["P"].astype(str)
    + "_Z:"
    + game_states["zone_id"].astype(str)
)

print(
    game_states[
        ["minute", "second", "team", "type", "delta_G", "P", "zone_id", "state_id"]
    ].head(15)
)

StatsBomb treats coordinates as relative from the perspective of the attacking team. So for whichever team has the ball, the opponent's goal is always at x=120.

It has been checked that there are no x or y values for the location that are outside [0, 120].

location data has been split into two columns x and y for ease of use.
